# 03 · Unidad de análisis y punto de predicción — Muerte fetal, MIMIC-IV (M3)

| Campo | Contenido |
|---|---|
| Secuencia del framework | 01: E1 · E2 · E3 · P1 → 02: E4 → **03: E5 · E6 → R3** → 04: R3 · E2 · E3 · P1 · E4 → 05: E5 · E6 · E7 · P2 → 06: P3 · clase · E11 |
| Qué hace | Construye o verifica la unidad de análisis (E5), mide qué se pierde en cada punto de predicción candidato (E6, grupo E de la auditoría) y cuántas muertes fetales ya están presentes en el punto elegido (grupo G). Con eso decide si tiene sentido un modelo al ingreso + 6 h |
| Etapas y puertas | E5 · E6 → retorno R3 |
| Decisiones | D4 (unidad), D6 (punto de predicción), D7 (tiempo cero y ventana), D8 (admisiones cortas) y el retorno R3 |
| Entradas | `fetal_death_label_v2` (notebook 01) y las tablas de MIMIC-IV `admissions`, `labevents`, `prescriptions`, `procedures_icd` |
| Tablas del esquema mimiciv | `unit_of_analysis_v2` (`sql/00_mimic_layer/01_unit_of_analysis_es.sql`) |
| Scripts | `sql/02_audit_counts/audit_counts_timing_es.sql` y `audit_counts_event_timing_es.sql` (solo lectura) |
| Salidas | `data/outputs_03_unit_and_prediction_point.txt` (solo conteos agregados; de 1 a 4 se muestran como "<5") |
| Tiempo | Varios minutos: los grupos E y G leen `labevents` y `prescriptions` |
| Anterior / siguiente | `02_count_audit` / `04_antenatal_count_audit` |
| Versión en inglés | `03_unit_and_prediction_point.ipynb` (mismo código) |

> **Reconstruir las tablas.** Las tablas del esquema `mimiciv` que usa este notebook se leen como están y solo se corren sus verificaciones. Si una tabla no existe, se construye con su script SQL. Para reconstruirla (por ejemplo, si cambia un script de `sql/`), poner **`REBUILD = True`** en la primera celda de código y volver a correr el notebook.

In [1]:
# Setup: paths, language, presentation helpers and the database connection
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
# Language of the outputs: "es" in the Spanish notebook, "en" in the English one (the only line that differs).
LANG = "es"
import importlib
import libraries.i18n_es, libraries.i18n, libraries.presentation, libraries.sql_runner
for _mod in (libraries.i18n_es, libraries.i18n, libraries.presentation, libraries.sql_runner):
    importlib.reload(_mod)                       # a running kernel picks up changes of the libraries
from libraries.presentation import setup, tr, say, show, show_results, check, decision, notebook_summary, write_outputs
from libraries.sql_runner import run_script, ensure_table, table_exists
from libraries.database import get_connect_database
setup(LANG)

REBUILD = False      # True: rebuild the mimiciv tables of this notebook from their SQL scripts
SQL = ROOT / "sql"
OUT_FILE = ROOT / "data" / "outputs_03_unit_and_prediction_point.txt"
engine = get_connect_database()

## E5 · Unidad de análisis — un registro por admisión, agrupado por paciente (decisión D4)

La tabla `unit_of_analysis_v2` es el adaptador de MIMIC-IV de la etapa E5: una fila por admisión del universo, con las columnas del contrato de datos (`unit_id`, `group_id`, `t0`, `t_pred` = ingreso + 6 h, `t_end`), el período, el orden de las admisiones y las cadenas de embarazo de 30, 90 y 180 días. No lleva nada del desenlace.

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| La tabla reproduce la auditoría y concuerda con la tabla de etiqueta | D4 queda fija: un registro por hospitalización de parto; toda partición agrupada por paciente | 25.736 unidades de 15.084 pacientes |
| Un conteo o una columna difiere | Revisar `sql/00_mimic_layer/01_unit_of_analysis_es.sql` y reconstruir | — |

In [2]:
# Stage S5: unit of analysis (task T17): mimiciv.unit_of_analysis_v2
res17 = ensure_table(engine, "unit_of_analysis_v2", SQL / "00_mimic_layer" / "01_unit_of_analysis.sql", REBUILD, log=say)
show_results(res17, ["Q-T17a integrity", "Q-T17b pregnancy chains", "Q-T17c gap to the previous admission",
                     "Q-T17d gap to an earlier event admission", "Q-T17e consistency with the label table",
                     "Q-T17f overlaps and ties", "Q-T17g prediction point inside the admission"])
t17 = res17["Q-T17a integrity"].iloc[0]
chains = res17["Q-T17b pregnancy chains"].set_index("gap_limit")["n_chains"]
cons = res17["Q-T17e consistency with the label table"].iloc[0]
MISMATCH = ["n_mismatch_d8_flag", "n_mismatch_icd_era", "n_mismatch_anchor", "n_mismatch_delivery", "n_mismatch_los_hours"]
s5_ok = check([
    ("units (admissions of the universe)", int(t17.n_units), 25736),
    ("patients", int(t17.n_patients), 15084),
    ("delivery hospitalizations", int(t17.n_delivery_hosp), 19425),
    ("first admissions (one per patient)", int(t17.n_first_admission), 15084),
    ("pregnancy chains, 30 days", int(chains.get(30, -1)), 23257),
    ("pregnancy chains, 90 days", int(chains.get(90, -1)), 22080),
    ("pregnancy chains, 180 days", int(chains.get(180, -1)), 21485),
    ("admissions only in one of the two tables (unit, label)", int(cons.n_unit_only) + int(cons.n_label_only), 0),
    ("mismatches with the label table (D8 flag, ICD era, anchor group, delivery flag, length of stay)",
     int(sum(int(cons[c]) for c in MISMATCH)), 0),
], "S5 · Unit of analysis: reference counts")
decision("S5 · Decision D4: unit of analysis", [
    ("record", "one row per admission", "one record per delivery hospitalization (population D3)"),
    ("grouping", f"{int(t17.n_patients):,} patients", "every validation split is grouped by patient"),
    ("same pregnancy", "90-day pregnancy chains", "sensitivity analysis"),
    ("checks of the table", "reproduced" if s5_ok else "DIFFERS", "on to S6" if s5_ok else "review the script before going on")])

construyendo mimiciv.unit_of_analysis_v2 con 01_unit_of_analysis.sql
01_unit_of_analysis.sql: 12 sentencias corridas (construcción y verificaciones), 0 omitidas, 0 errores, 4.1 s


,filas,unidades,pacientes,hospitalizaciones de parto,primeras admisiones,universo,tiempos faltantes,estancia negativa,terminan antes de 6 h
1,"25,736","25,736","15,084","19,425","15,084","25,736",0,<5,365


,intervalo máximo (días),cadenas,cadenas con 2 o más admisiones,máximo de admisiones en una cadena,cadenas con 2 o más partos
1,30,"23,257","2,034",9,20
2,90,"22,080","2,787",11,29
3,180,"21,485","3,202",11,31


,intervalo con la admisión anterior,admisiones,eventos
1,0 primera admisión de la paciente,"15,084",93
2,1 menos de 90 días,"3,649",21
3,2 90-179 días,595,<5
4,3 180-364 días,415,<5
5,4 365 días o más,"5,993",37


,intervalo con un evento anterior,admisiones,eventos
1,0 sin admisión anterior con evento,"25,626",148
2,1 menos de 90 días,17,5
3,3 180-364 días,15,0
4,4 365 días o más,78,<5


,solo en la unidad,solo en la etiqueta,difiere la marca D8,difiere la era CIE,difiere el grupo de año,difiere la marca de parto,difiere la estancia
1,0,0,0,0,0,0,0


,con admisión anterior,empieza antes de que termine la anterior,misma hora de ingreso que la anterior
1,"10,652",7,0


,marca de parto v2,admisiones,terminan antes de 6 h,estancia media (h),estancia mediana (h),admisiones medias por paciente
1,No,"6,311",362,63.5,39.8,3.22
2,Sí,"19,425",<5,90.5,70.8,2.09
3,,"25,736",365,83.8,66.4,2.37


,conteo,cargado,esperado,resultado
1,unidades (admisiones del universo),"25,736","25,736",reproducido
2,pacientes,"15,084","15,084",reproducido
3,hospitalizaciones de parto,"19,425","19,425",reproducido
4,primeras admisiones (una por paciente),"15,084","15,084",reproducido
5,"cadenas de embarazo, 30 días","23,257","23,257",reproducido
6,"cadenas de embarazo, 90 días","22,080","22,080",reproducido
7,"cadenas de embarazo, 180 días","21,485","21,485",reproducido
8,"admisiones solo en una de las dos tablas (unidad, etiqueta)",0,0,reproducido
9,"diferencias con la tabla de etiqueta (marca D8, era CIE, grupo de año, marca de parto, estancia)",0,0,reproducido


,revisión,resultado,decisión
1,registro,una fila por admisión,un registro por hospitalización de parto (población D3)
2,agrupación,"15,084 pacientes",toda partición de la validación se agrupa por paciente
3,mismo embarazo,cadenas de embarazo de 90 días,análisis de sensibilidad
4,verificaciones de la tabla,reproducido,se pasa a E6


## E6 · Punto de predicción — qué pierde cada punto candidato (grupo E; decisiones D6, D7 y D8)

Las mujeres con muerte fetal están menos tiempo en el hospital que las demás, así que un punto de predicción tardío pierde eventos. El grupo E mide la duración de la estancia, los eventos que siguen hospitalizados en cada punto candidato y cuándo ocurren el primer laboratorio, el registro en urgencias, la UCI y los procedimientos de parto.

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| A las 24 h se pierde el 13,2% de los eventos estrictos frente al 0,15% de los no eventos | D6: punto de predicción al ingreso + 6 h (12 h y 24 h como sensibilidad) | A las 6 h se conservan los 129 eventos estrictos |
| El primer laboratorio se toma alrededor del ingreso | D7: tiempo cero = ingreso; ventana desde 24 h antes del ingreso hasta el punto de predicción | La mitad de los primeros valores es anterior a la hora de ingreso |
| 3 admisiones terminan antes de 6 h, ninguna es evento | D8: se excluyen y se reportan aparte | 19.437 admisiones en el punto de predicción |

In [3]:
# Stage S6: candidate prediction points (group E of the count audit, read-only script)
tim = run_script(engine, SQL / "02_audit_counts" / "audit_counts_timing.sql", log=say)
show_results(tim, ["Q-E0 population check", "Q-E1 length of stay", "Q-E2 eligibility at 24 hours",
                   "Q-E2b remaining at other prediction points", "Q-E3a ED registration before admission",
                   "Q-E3b first laboratory value", "Q-E3c first ICU admission", "Q-E4 delivery and induction procedure dates"])
e2b = tim["Q-E2b remaining at other prediction points"].set_index("hours_after_admission")
strict0 = int(e2b.loc[0, "n_strict_remaining"])
lost = {h: strict0 - int(e2b.loc[h, "n_strict_remaining"]) for h in e2b.index}
decision("S6 · Decisions D6, D7 and D8: prediction point", [
    ("strict events lost at 6, 12 and 24 h", f"{lost.get(6)}, {lost.get(12)} and {lost.get(24)} of {strict0}",
     "D6: prediction point = admission + 6 h (12 h and 24 h as sensitivity)"),
    ("first laboratory value and emergency registration (Q-E3)", "around the admission time; registration a median of about 3 h before",
     "D7: time zero = admission; window from 24 h before admission to the prediction point"),
    ("admissions that end before 6 h", f"{int(e2b.loc[0, 'n_remaining']) - int(e2b.loc[6, 'n_remaining'])} (events: {int(e2b.loc[0, 'n_events_remaining']) - int(e2b.loc[6, 'n_events_remaining'])})",
     "D8: excluded and reported apart")])

audit_counts_timing.sql: 20 sentencias corridas (solo verificaciones), 0 omitidas, 0 errores, 55.2 s


,población,eventos estrictos o parciales,eventos estrictos,marca de parto,estrictos con marca de parto
1,"19,440",142,129,"19,425",114


,grupo,admisiones,estancia faltante o no positiva,p05 (h),p25 (h),mediana (h),p75 (h),p95 (h),menos de 12 h,12 a 24 h,24 a 48 h,48 h o más
1,1 evento estricto,129,0,18.6,36.7,54.4,87.8,226.1,5,12,37,75
2,2 evento solo parcial,13,0,28.4,61.3,77.1,98.9,159.8,0,0,<5,10
3,3 no evento,"19,298",0,44.7,58.1,71,100.5,159.4,7,21,"1,497","17,773"
4,,"19,440",0,44.5,57.9,70.8,100.5,160.1,12,33,"1,537","17,858"


,alcance,admisiones,elegibles,murieron antes de 24 h,egresaron antes de 24 h,eventos,eventos perdidos,estrictos,estrictos perdidos,% de estrictos perdidos
1,Población D3 (marca de parto o código de mortinato),"19,440","19,395",<5,44,142,17,129,17,13.2
2,Solo la marca de parto,"19,425","19,383",<5,41,127,14,114,14,12.3
3,Cualquier admisión obstétrica,"25,736","23,625",5,"2,106",154,20,136,20,14.7


,horas desde el ingreso,admisiones restantes,eventos restantes,estrictos restantes,% de estrictos conservados
1,0,"19,440",142,129,100
2,6,"19,437",142,129,100
3,12,"19,428",137,124,96.1
4,24,"19,395",125,112,86.8
5,36,"19,144",108,97,75.2
6,48,"17,858",85,75,58.1


,grupo,admisiones,con registro en urgencias,mediana de horas antes del ingreso,p95 de horas antes del ingreso,urgencias más de 24 h antes,urgencias después de la hora de ingreso
1,1 evento estricto,129,20,3.4,12.8,0,0
2,2 evento solo parcial,13,0,,,0,0
3,3 no evento,"19,298",427,2.6,14.8,0,<5
4,,"19,440",447,2.6,14.8,0,<5


,grupo,admisiones,sin valor de laboratorio,p05 (h),mediana (h),p95 (h),primer laboratorio antes del ingreso,primer laboratorio después de 24 h
1,1 evento estricto,129,25,-7.5,0,3.3,52,<5
2,2 evento solo parcial,13,<5,-7.2,-0.8,1.2,8,0
3,3 no evento,"19,298","3,418",-7.8,0.1,5,"7,544",71
4,,"19,440","3,445",-7.8,0.1,4.9,"7,604",72


,grupo,admisiones,con estancia en UCI,UCI en las primeras 24 h,mediana de horas hasta la UCI,p95 de horas hasta la UCI
1,1 evento estricto,129,8,5,11.1,247.4
2,2 evento solo parcial,13,0,0,,
3,3 no evento,"19,298",197,110,20.6,422.5
4,,"19,440",205,115,19.9,417.5


,grupo,admisiones,sin procedimiento de parto,parto antes de la fecha de ingreso,parto el día 0,parto el día 1,parto el día 2 o después,con inducción,inducción el día 0,inducción el día 1 o después
1,1 evento estricto,129,21,0,36,51,21,43,14,29
2,2 evento solo parcial,13,<5,0,5,5,<5,<5,<5,<5
3,3 no evento,"19,298","4,293",21,"8,551","4,733","1,700","6,770","4,602","2,113"
4,,"19,440","4,315",21,"8,592","4,789","1,723","6,817","4,619","2,143"


,revisión,resultado,decisión
1,"eventos estrictos perdidos a las 6, 12 y 24 h","0, 5 y 17 de 129",D6: punto de predicción = ingreso + 6 h (12 h y 24 h como sensibilidad)
2,primer laboratorio y registro en urgencias (Q-E3),alrededor de la hora de ingreso; registro en urgencias una mediana de unas 3 h antes,D7: tiempo cero = ingreso; ventana desde 24 h antes del ingreso hasta el punto de predicción
3,admisiones que terminan antes de 6 h,3 (eventos: 0),D8: se excluyen y se reportan aparte


## E6 · Punto de predicción — ¿las muertes ya están presentes al ingreso + 6 h? (grupo G)

La etiqueta es un código de egreso y no tiene momento. El grupo G cuenta los indicios que pueden ubicar cada muerte en el tiempo: un código de muerte intrauterina (P1), un código de muerte fetal como diagnóstico principal (P2), un estudio de laboratorio de muerte fetal con su hora (P3), mifepristona o misoprostol con su hora (P4) y un procedimiento de parto el día del ingreso (P5). Las clases 1 y 2 de Q-G6 son las muertes ya presentes en el punto de predicción. La pregunta de la etapa E6 es qué se puede predecir en ese punto; si la mayoría de las muertes ya están presentes, un modelo reconocería muertes que ya ocurrieron y fallaría el criterio C1 de la puerta P4.

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| Menos de la mitad de las muertes ya presentes en T | El modelo al ingreso + 6 h sigue a E7 (predictores) | — |
| Más de la mitad ya presentes en T | El modelo al ingreso + 6 h no se ajusta: su destino (C1) se conoce por la auditoría. **Retorno R3**: otro punto de predicción (notebook 04) | 73 de 129 (56,6%): **retorno R3** |

In [4]:
# Stage S6: timing of the fetal-death events (group G of the count audit, read-only script; several minutes)
evt = run_script(engine, SQL / "02_audit_counts" / "audit_counts_event_timing.sql", log=say)
show_results(evt, ["Q-G0 population check", "Q-G1 untimed proxies (strict events)", "Q-G2 candidate work-up laboratory items",
                   "Q-G3 first work-up laboratory value relative to T", "Q-G4 mifepristone and misoprostol",
                   "Q-G5 proxies in events and non-events", "Q-G6 candidate timing classes (strict events)"])
g6 = evt["Q-G6 candidate timing classes (strict events)"]
cls = g6.dropna(subset=["timing_class"]).set_index("timing_class")["n_events"]
n_strict_t = int(g6.loc[g6["timing_class"].isna(), "n_events"].iloc[0])
present = int(cls[[c for c in cls.index if "present at T" in c]].sum())
share = present / n_strict_t
check([("strict events already present at the prediction point (classes 1 and 2 of Q-G6)", present, 73),
       ("strict events at the prediction point", n_strict_t, 129)],
      "S6 · Events already present at the prediction point: reference counts (catalogue of models)")
decision("S6 · Decision: what can be predicted at admission + 6 h", [
    ("strict events already present at the prediction point", f"{present} of {n_strict_t} ({100 * share:.1f}%)",
     "more than half: a model at admission + 6 h would fail criterion C1" if share > 0.5 else "less than half: on to S7"),
    ("model at admission + 6 h", "not fitted" if share > 0.5 else "goes on",
     "return R3 (another prediction point): antenatal prediction, notebook 04" if share > 0.5 else "stage S7"),
    ("returns used", "R3 (1 of 2 allowed)" if share > 0.5 else "none", "—")])

audit_counts_event_timing.sql: 20 sentencias corridas (solo verificaciones), 0 omitidas, 0 errores, 48.8 s


,admisiones,estrictos,eventos con parciales
1,"19,437",129,142


,era CIE,familia de código,eventos,diagnóstico principal,con registro en urgencias
1,solo CIE-10,1 solo código de muerte intrauterina,<5,0,0
2,solo CIE-10,2 solo código de desenlace de mortinato,27,0,9
3,solo CIE-10,3 ambas familias,43,38,5
4,solo CIE-10,,71,38,14
5,solo CIE-9,1 solo código de muerte intrauterina,<5,<5,0
6,solo CIE-9,2 solo código de desenlace de mortinato,20,0,<5
7,solo CIE-9,3 ambas familias,34,26,<5
8,solo CIE-9,,58,27,6
9,,,129,65,20


,itemid,examen,fluido,categoría,eventos estrictos con el examen,no eventos con el examen
1,"51,212",Fetal Hemoglobin,Blood,Hematology,57,132
2,"52,351",Fetal Hgb,Other Body Fluid,Hematology,0,0


,itemid,examen,grupo,admisiones con valor,hasta T,de T a 24 h,después de 24 h,mediana de horas desde el ingreso
1,"51,212",Fetal Hemoglobin,1 evento estricto,57,49,<5,<5,1.6
2,"51,212",Fetal Hemoglobin,2 evento solo parcial,<5,0,<5,0,7.7
3,"51,212",Fetal Hemoglobin,3 no evento,132,37,60,35,12.2


,grupo,admisiones,mifepristona,mifepristona hasta T,misoprostol,misoprostol hasta T,mediana de horas hasta la mifepristona
1,1 evento estricto,129,0,0,6,<5,None
2,2 evento solo parcial,13,0,0,0,0,None
3,3 no evento,"19,295",0,0,468,46,None
4,,"19,437",0,0,474,47,None


,grupo,admisiones,P1 código de muerte intrauterina,P2 diagnóstico principal,P3 laboratorio hasta T,P3 laboratorio después de T,P4 mifepristona hasta T,P4 misoprostol hasta T,P5 parto el día 0
1,1 evento estricto,129,82,65,49,8,0,<5,36
2,2 evento solo parcial,13,0,0,0,<5,0,0,5
3,3 no evento,"19,295",0,0,37,95,0,46,"8,550"


,clase temporal,eventos,CIE-10,con código de pérdida temprana,22 semanas o más,estudio solo después de T
1,"1 presente en T, con hora",49,29,<5,26,0
2,"2 presente en T, por código",24,14,0,14,<5
3,"3 anteparto, hora desconocida",14,<5,0,<5,<5
4,4 indeterminado,42,24,7,13,5
5,,129,71,10,55,8


,conteo,cargado,esperado,resultado
1,eventos estrictos ya presentes en el punto de predicción (clases 1 y 2 de Q-G6),73,73,reproducido
2,eventos estrictos en el punto de predicción,129,129,reproducido


,revisión,resultado,decisión
1,eventos estrictos ya presentes en el punto de predicción,73 de 129 (56.6%),más de la mitad: un modelo al ingreso + 6 h fallaría el criterio C1
2,modelo al ingreso + 6 h,no se ajusta,"retorno R3 (otro punto de predicción): predicción prenatal, notebook 04"
3,retornos usados,R3 (1 de 2 permitidos),—


## Resumen del notebook

Indicadores principales de este notebook y el paso siguiente.

In [5]:
# Summary of notebook 03
notebook_summary([
    ("unit of analysis (D4)", "delivery hospitalization; every split grouped by patient"),
    ("checks of the unit table", lambda: "reproduced" if s5_ok else "DIFFERS"),
    ("prediction point (D6)", "admission + 6 h"),
    ("strict events lost at 24 h", lambda: f"{lost.get(24)} of {strict0}"),
    ("strict events already present at admission + 6 h", lambda: f"{present} of {n_strict_t} ({100 * share:.1f}%)"),
    ("model at admission + 6 h", lambda: "not fitted: criterion C1 anticipated by the audit" if share > 0.5 else "goes on to S7"),
    ("next step", lambda: "return R3: antenatal prediction point, notebook 04" if share > 0.5 else "predictors (S7)"),
])
write_outputs(OUT_FILE, "Notebook 03 (S5, S6, return R3), fetal death MIMIC-IV (M3); aggregate output only")

,valor
indicador,
unidad de análisis (D4),hospitalización de parto; toda partición agrupada por paciente
verificaciones de la tabla de unidad,reproducido
punto de predicción (D6),ingreso + 6 h
eventos estrictos perdidos a las 24 h,17 de 129
eventos estrictos ya presentes al ingreso + 6 h,73 de 129 (56.6%)
modelo al ingreso + 6 h,no se ajusta: la auditoría anticipó el criterio C1
paso siguiente,"retorno R3: punto de predicción prenatal, notebook 04"


salida agregada escrita: data/outputs_03_unit_and_prediction_point.txt
